## Exploring predictions

In [ ]:
import os
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import torch

from utils import utils
from data_builder import data_methods
from data_builder import build_tags
from data_builder import data_loader
from model_builder import build_model
import trainer.metrics as metrics_module
from predictor import inference
from torchvision.transforms import ToTensor
from torchvision import transforms

from data_builder import read_landsat

# from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

In [ ]:
# GET config
EXP_NAME = "exp_001"
REWRITE = True

# config = experiment_config.get_config(EXP_NAME)
config = utils.get_config(EXP_NAME)
config["mode"] = "inference"
config["batch_size"] = config["inference"]["batch_size"]

directory_paths = utils.get_directories()
SAVE_MODEL_DIRECTORY = directory_paths["save_model_dir"]
DATA_DIRECTORY = directory_paths["data_dir"]
FIGURE_DIRECTORY = directory_paths["figures_dir"]
PREDICTIONS_DIRECTORY = directory_paths["predictions_dir"]
LANDSAT_DIRECTORY = directory_paths["landsat_dir"]
MOSAICS_DIRECTORY = directory_paths["mosaics_dir"]

In [ ]:
# GET THE DATA

# config["inference_region"] = (-7, -6, 106, 107)
config["inference_region"] = (30, 39.9999, 70, 79.99999)

# set and get important config
(lat_s_bound, lat_n_bound, lon_w_bound, lon_e_bound) = read_landsat.get_landsat_bounds(
    config, region=config["inference_region"]
)

# load the model
model = build_model.get_model(config)

for year in np.arange(2020, 2021):
    print(" --- " + str(year) + "---")
    config["inference_years"] = (year,)
    filenames_list = []

    for latfile in np.arange(
        lat_s_bound + config["tile_len_deg"],
        lat_n_bound + config["tile_len_deg"],
        config["tile_len_deg"],
    ):
        for lonfile in np.arange(lon_w_bound, lon_e_bound, config["tile_len_deg"]):
            # CHECK IF LANDSAT FILE EXISTS
            config["tile"] = (
                latfile - config["tile_len_deg"],
                latfile,
                lonfile,
                lonfile + config["tile_len_deg"],
            )
            landsat_file = read_landsat.get_input_filename(
                config["inference_years"], (latfile,), (lonfile,), config
            )

            # CHECK IF LANDSAT FILE EXISTS
            if os.path.isfile(LANDSAT_DIRECTORY + landsat_file[0] + ".tif") is False:
                continue

            # TODO: check if landsat tile is all water, if so, create prediction file of all NODATA

            # CHECK IF PREDICTION FILE ALREADY EXISTS
            predictions_filename = (
                config["exp_name"] + "_predictions_" + landsat_file[0]
            )
            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            if (
                os.path.isfile(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
                and REWRITE is False
            ):
                continue
            print(landsat_file[0])

            # GET THE SAMPLE TAGS
            tags, __ = build_tags.get_tags(config)
            if len(tags[0]) == 0:
                continue

            # MAKE PREDICTIONS and SAVE AS TIFF
            ds_inf = data_loader.CustomData(config, tags)
            inf_loader = torch.utils.data.DataLoader(
                ds_inf,
                batch_size=None,
                batch_sampler=None,
                shuffle=False,
                drop_last=False,
                pin_memory=config["inference"]["pin_memory"],
                num_workers=config["inference"]["num_workers"],
            )
            hfi_predict, hfi_labels, latlon_bounds = inference.make_predictions(
                config, model, tags, inf_loader
            )

            meta_data = inference.save_predictions_tif(
                hfi_predict,
                PREDICTIONS_DIRECTORY + predictions_filename + ".tif",
                latlon_bounds=latlon_bounds,
            )

            filenames_list.append(PREDICTIONS_DIRECTORY + predictions_filename + ".tif")
            print("\n")

    # TILE THE PREDICTIONS TOGETHER
    mosaic_filename = (
        MOSAICS_DIRECTORY
        + config["exp_name"]
        + "_"
        + str(config["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    mosaic, mosaic_trans = inference.create_mosaic(filenames_list)
    meta_data = inference.save_predictions_tif(
        mosaic, mosaic_filename, trans=mosaic_trans
    )
    print("mosaic saved.")